Привет, Даниил. Да, удобно на ты :)

Буду благодарен за любым обратную связь даже по мелким недочетам: нейминг, какой-то странное использование написание методов и тд. Перехожу на Python c Dart, было бы круто узнать что-то новое

# Вопросы:
1) Корректно ли заменять, например, ссылки на такую маску - [LINK], чтобы модель понимала контекст, что в данном месте есть ссылка? Или это приведет к ошибкам в выводе?
2) Корректно ли сравнивать 

# Исправления в коде вне ipynb:
1) utils.py
```python
torch.device("gpu") -> return torch.device("cuda")
```

In [1]:
%load_ext autoreload
%autoreload 2
from src.data_utils import DataUtils
import yaml
import pandas as pd
from src.utils import my_device
from src.next_token_dataset import NextTokenDataset
from transformers import BertTokenizerFast
from torch.utils.data import DataLoader
import torch
from src.lstm_model import LstmModel
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM
from src.transformer import transformer_make_prediction
import evaluate

/Users/slermo/development/projects/1_py/ynd_sem2_lstm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
# Загрузка конфига
with open("configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)


In [3]:
# Создание csv файлов, если есть только исходники
DataUtils.samples_create(config['dataset'])

1280398 160050 160050


In [14]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

df_train = pd.read_csv(config['dataset']['path'] + '/train.csv')
df_val = pd.read_csv(config['dataset']['path'] + '/val.csv')

train_dataset = NextTokenDataset(df_train["text"].tolist(), tokenizer, max_len=16)
val_dataset = NextTokenDataset(df_val["text"].tolist(), tokenizer, max_len=16)

# num_workers=0 - чтобы не было дедлоков
# The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=0)


In [15]:
for batch in train_loader:
    print(batch["input_ids"].shape)
    break

torch.Size([256, 15])


In [ ]:
from src.eval_lstm_pipeline import eval_lstm_pipeline
model = LstmModel(vocab_size=tokenizer.vocab_size, hidden_dim=128).to(my_device())

eval_lstm_pipeline(
    config=config,
    model=model,
    tokenizer=tokenizer,
    train_loader=train_loader,
    val_loader=val_loader
)

KeyboardInterrupt: 

In [ ]:
generator = pipeline("text-generation", model="distilgpt2")
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

model_name = "distilgpt2"          # лёгкая версия GPT-2
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Device set to use mps:0


In [24]:
# Тестирование моделей
df_test = pd.read_csv(config['dataset']['path'] + '/test.csv')

rouge = evaluate.load("rouge")

for i in range(10):
    text = df_test.iloc[i]["text"]
    words = text.split()
    split_point = len(words) * 3 // 4
    prompt = words[:split_point]
    prompt_str = " ".join(prompt)
    reference = " ".join(words[split_point:]) # Часть для сравнения с генерацией

    print('PROMPT: ' + prompt_str)
    # LSTM
    model.eval()
    prompt_tokens = tokenizer.encode(prompt_str, return_tensors='pt').to(my_device())
    with torch.no_grad():
            lstm_generated_tokens = model.generate(
                prompt_tokens, 
                max_new_tokens=20,
            )
    lstm_text = tokenizer.decode(lstm_generated_tokens[0], skip_special_tokens=True)
    lstm_rouge_scores = rouge.compute(
            predictions=[lstm_text[len(prompt_str):].strip()], 
            references=[reference]
    )
    print(f"LSTM result: {lstm_text}")
    print(f"ROGUE-1 {lstm_rouge_scores['rouge1']:.4f}")
    print(f"ROGUE-2 {lstm_rouge_scores['rouge2']:.4f}")
    print(f"ROGUE-L {lstm_rouge_scores['rougeL']:.4f}")

    trns_output = generator(text, max_new_tokens=20, do_sample=True, top_k=50, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
    trns_text = trns_output[0]["generated_text"]  # Извлекаем текст из первого результата
    trns_gen_only = trns_text[len(prompt_str):].strip()

    trns_rouge_scores = rouge.compute(
            predictions=[trns_gen_only], 
            references=[reference]
        )

    print('TNSF result: '+ trns_text.replace("\n", " ").strip())
    print(f"ROGUE-1 {trns_rouge_scores['rouge1']:.4f}")
    print(f"ROGUE-2 {trns_rouge_scores['rouge2']:.4f}")
    print(f"ROGUE-L {trns_rouge_scores['rougeL']:.4f}")

    # Модель distilgpt2
    transformer_make_prediction(generator,tokenizer, text)
    print('-'*80)




[autoreload of src.lstm_model failed: Traceback (most recent call last):
  File "/Users/slermo/development/projects/1_py/ynd_sem2_lstm/.venv/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/Users/slermo/development/projects/1_py/ynd_sem2_lstm/.venv/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 500, in superreload
    update_generic(old_obj, new_obj)
  File "/Users/slermo/development/projects/1_py/ynd_sem2_lstm/.venv/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    update(a, b)
  File "/Users/slermo/development/projects/1_py/ynd_sem2_lstm/.venv/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 349, in update_class
    if update_generic(old_obj, new_obj):
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/slermo/development/projects/1_py/ynd_sem2_lstm/.venv/lib/python3.12/site-packages/IPython/extensions/autor

PROMPT: joejonas1fan1 glad to hear it im alright just cant sleep lol really tired but my nerves wont
LSTM result: joejonas1fan1 glad to hear it im alright just cant sleep lol really tired but my nerves wont
ROGUE-1 0.0000
ROGUE-2 0.0000
ROGUE-L 0.0000


NameError: name 'generator' is not defined